Rota Maker

Times:
- Open Start = 8
- Close Finish = 11
- Busy Hours = 12-2, 5-7
- Quiet Hours = 3-4
- Shift Size = 4-8hrs

In [54]:
from dataclasses import dataclass
from datetime import date, time, datetime, timedelta #datetime is a specific point in time and timedelta is just a period of time
import pandas as pd

In [55]:
class Employee:
  _id_counter = 1 #Class variable

  def __init__(self, name, contract_hours, max_shifts_per_week, job_role, departments):
    self.id = Employee._id_counter #Make a unique ID
    Employee._id_counter += 1 #Increase counter
    self.name = name #Who is it
    self.contract_hours = contract_hours #Min hours
    self.assigned_hours = 0 #How many hours are they working
    self.max_shifts_per_week = max_shifts_per_week #How many days are they in
    self.job_role = job_role #Supervisor or FOH Assistant
    self.departments = set(departments) #Hosp or cinema

  @property
  def remaining(self) -> float:
    return self.contract_hours - self.assigned_hours

@dataclass(frozen=True) #Frozen so values cannot be changed once created
class Shift:
  day: date
  start: time
  end: time
  hours: float
  department: str
  required: bool = True


In [56]:
def make_shifts(start_day: date) -> list[Shift]:
  shifts = []
  for i in range(7):
    shift_date = start_day + timedelta(days=i) #Makes a weeks worth of shifts
    shifts.append(Shift(shift_date, time(8, 0), time(16, 0), 7.5, department="Hospitality", required=True)) #open
    shifts.append(Shift(shift_date, time(15, 0), time(23, 0), 7.5, department="Hospitality", required=True)) #close
    shifts.append(Shift(shift_date, time(11, 0), time(19, 0), 7.5, department="Hospitality", required=False)) #mid
    shifts.append(Shift(shift_date, time(12, 0), time(20, 0), 7.5, department="Hospitality", required=False)) #mid
    shifts.append(Shift(shift_date, time(9, 0), time(17, 0), 7.5, department="Cinema", required=True)) #open
    shifts.append(Shift(shift_date, time(15, 0), time(23, 0), 7.5, department="Cinema", required=True)) #close
  return shifts

In [57]:
def build_combined_rota(employees: list[Employee], shifts: list[Shift]):
  rota = []
  shifts_assigned = {person.id: 0 for person in employees} #makes a dictionary assigning an employees id to how many shifts theyre working
  worked_days = {person.id: set() for person in employees} #makes a dictionary assigning an employees id to a set with the days they've worked in the week
  last_end_dt = {person.id: None for person in employees} #makes a dictionary assigning an employees id to the last time they were at work


  def shift_datetimes(shift: Shift) -> tuple:
    start_dt = datetime.combine(shift.day, shift.start) #The shift date and start time
    end_dt = datetime.combine(shift.day, shift.end) #The shift date and end time
    return start_dt, end_dt


  def can_take_shift(person: Employee, shift: Shift) -> bool: #Returns yes or no whether the person can take a shift or not
    if shift.department not in person.departments: #Checks the person can work in that area
      return False
    if shifts_assigned[person.id] >= person.max_shifts_per_week: #Checks if they've hit their max shifts
      return False
    if shift in worked_days[person.id]: #Checks the person hasn't already working this day
      return False
    prev_end = last_end_dt[person.id]
    if prev_end is not None: #Checks the person has worked a shift
      start_dt, _ = shift_datetimes(shift) #Take start_dt, ignore end_dt
      if start_dt - prev_end < timedelta(hours=12): #Checks there's a gap of 12 hours between this new shift and their last one
        return False
    return True


  def assign(shift: Shift, chosen: Employee): #Assigns a shift to a person
    _, end_dt = shift_datetimes(shift)
    chosen.assigned_hours += shift.hours
    shifts_assigned[chosen.id] += 1
    worked_days[chosen.id].add(shift.day)
    last_end_dt[chosen.id] = end_dt
    rota.append((shift, chosen.name))


  def eligible_count(shift: Shift) -> int:
    return sum(1 for person in employees if can_take_shift(person, shift)) #Returns how many people can work a particular shift


  required_shifts = [s for s in shifts if s.required] #opens+closes
  optional_shifts = [s for s in shifts if not s.required] #optional mid shifts

  required_shifts.sort(key=eligible_count) #cover the hardest shifts first

  for shift in required_shifts: #Assigns the open/close shifts first
    eligible = [person for person in employees if can_take_shift(person, shift)]
    if not eligible:
      raise ValueError(f"ERROR: Cannot cover open/close -> {shift}")
    chosen = max(eligible, key=lambda person: (person.remaining, -shifts_assigned[person.id]))
    assign(shift, chosen)

  total_deficit = sum(max(0.0, person.remaining) for person in employees)
  total_optional = sum(shift.hours for shift in optional_shifts)
  if total_deficit > total_optional:
    raise ValueError(f"ERROR: Need {total_deficit} more hours to cover contracted hours")

  for shift in optional_shifts:
    eligible = [person for person in employees if can_take_shift(person, shift) and person.remaining > 0]
    if not eligible:
      continue
    chosen = max(eligible, key=lambda person: person.remaining)
    assign(shift, chosen)

  return rota


In [58]:
def create_rota_table(rota: list[tuple[Shift, str]]):
  rows = []
  for shift, name in rota:
    rows.append({
        "Name":name,
        "Date":shift.day,
        "Shift": f"{shift.start.strftime("%H:%M")}-{shift.end.strftime("%H:%M")}",
        "Hours":shift.hours
    })

  df = pd.DataFrame(rows) #Makes table
  rota_table = df.pivot(index="Name", columns="Date", values="Shift") #Assigns information to table
  rota_table.columns = [d.strftime("%a %d") for d in rota_table.columns] #Makes date look pretty
  rota_table = rota_table.fillna("-----------") #Fills gaps with ---
  rota_table.sort_index(axis=1) #Sorts by date on the top
  totals = df.groupby("Name")["Hours"].sum() #Finds the total hours per person
  rota_table["Totals"] = totals #Adds the totals column

  return rota_table

In [59]:
supervisors = [
    Employee("Aleks", 37.5, 5, "Supervisor", {"Hospitality"}),
    Employee("George", 21, 3, "Supervisor", {"Cinema"}),
    Employee("Millie", 21, 3, "Supervisor", {"Cinema"}),
    Employee("Ben", 21, 3, "Supervisor", {"Hospitality", "Cinema"}),
    Employee("V", 21, 3, "Supervisor", {"Hospitality"}),
    Employee("Kyle", 21, 3, "Supervisor", {"Hospitality", "Cinema"}),
    Employee("Luke C", 21, 3, "Supervisor", {"Hospitality", "Cinema"}),
    Employee("Lorna", 37.5, 5, "Manager", {"Hospitality"}),
    Employee("Catherine", 37.5, 5, "Manager", {"Cinema"})
]

shifts = make_shifts(date(2026, 3, 6))
combined_rota = build_combined_rota(supervisors, shifts)
hosp_rota = [(shift, name) for (shift, name) in combined_rota if shift.department == "Hospitality"]
cinema_rota = [(shift, name) for (shift, name) in combined_rota if shift.department == "Cinema"]

print("---------------------------HOSPITALITY---------------------------")
print("\n", create_rota_table(hosp_rota), "\n")
print("-----------------------------CINEMA------------------------------")
print("\n", create_rota_table(cinema_rota), "\n")

---------------------------HOSPITALITY---------------------------

             Fri 06       Sat 07       Sun 08       Mon 09       Tue 10  \
Name                                                                     
Aleks  08:00-16:00  08:00-16:00  08:00-16:00  -----------  08:00-16:00   
Ben    -----------  -----------  -----------  -----------  -----------   
Kyle   -----------  -----------  -----------  15:00-23:00  -----------   
Lorna  15:00-23:00  15:00-23:00  15:00-23:00  -----------  15:00-23:00   
V      -----------  -----------  -----------  08:00-16:00  -----------   

            Wed 11       Thu 12  Totals  
Name                                     
Aleks  -----------  08:00-16:00    37.5  
Ben    08:00-16:00  11:00-19:00    15.0  
Kyle   -----------  -----------     7.5  
Lorna  -----------  15:00-23:00    37.5  
V      15:00-23:00  12:00-20:00    22.5   

-----------------------------CINEMA------------------------------

                 Fri 06       Sat 07       Sun 08 